# Lab 12 — Phase 3

A* (offline) → execute one segment → 360° Bayes localization → update the
believed pose → re-plan from the believed pose → next segment.

Heading 0° = facing +y (Lab 9 IMU convention). The robot starts at
`WAYPOINTS[0] = (-4, -3)` physically pointing at +y; one `RESET_YAW`
at the start of `run_mission` makes IMU yaw == world heading for the
whole run.

Localization: 18 ToF readings over 360°, front-facing ToF only — same
as Lab 11 (matches `world.yaml: observations_count: 18`).

## 1. Imports

In [1]:
import asyncio
import math
import os
import pathlib
import time

import matplotlib.pyplot as plt
import numpy as np

# Localization framework (FastRobots-sim-release).
from notebook_utils import *
from Traj import *
from utils import load_config_params
from localization import Mapper
from localization_extras import Localization
from notebook_utils import GET_GUI, START_PLOTTER, STOP_PLOTTER, RESET_PLOTTER, get_logger

# BLE (same notebooks/ dir).
from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD

# A* planner (same dir).
from astar import (
    build_occupancy_grid, astar, cells_to_segments,
    world_to_cell, cell_to_world,
)

LOG = get_logger('lab12_phase3.log')
LOG.propagate = False

/Users/y1hhnn/.pyenv/versions/FastRobots_ble/lib/python3.13/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


2026-05-13 22:22:54,057 | INFO     |: Logger lab12_phase3.log initialized.


## 2. Map, waypoints, and grid build
Mirrors `lab12_plan.py` — same map, same robot footprint.

In [2]:
# Matches FastRobots-sim-release/config/world.yaml exactly (m -> ft).
MAP_SIM = {
    'outer': [
        (-5.5, -4.5), ( 6.5, -4.5), ( 6.5,  4.5), (-2.5,  4.5),
        (-2.5,  0.5), (-5.5,  0.5),
    ],
    'obstacles': [
        # Inner box (2 ft × 2 ft, world.yaml lines 7–10).
        [( 2.5, -0.5), ( 4.5, -0.5), ( 4.5,  1.5), ( 2.5,  1.5)],
        # Lower U-shape pillar jutting up from the bottom wall
        # (1 ft × 2 ft, world.yaml lines 11–13 — bottom edge shared with
        #  the outer bottom wall, so the closed obstacle polygon is a rectangle).
        [(-0.5, -4.5), ( 0.5, -4.5), ( 0.5, -2.5), (-0.5, -2.5)],
    ],
}

WAYPOINTS = [
    (-4, -3),  # 1. start (robot starts here facing +y)
    (-2, -1),  # 2
    ( 1, -1),  # 3
    ( 2, -3),  # 4
    ( 5, -3),  # 5
    ( 5, -2),  # 6
    ( 5,  3),  # 7
    ( 0,  3),  # 8
    ( 0,  0),  # 9. end
]

CELL_SIZE = 1.0
INFLATE   = 0
X_RANGE   = (-6.0, 8.0)
Y_RANGE   = (-5.0, 5.0)
FT_TO_M   = 0.3048
ROBOT_RADIUS_FT = math.hypot(0.18, 0.10) / 2 / FT_TO_M    # ≈ 0.338 ft

grid, origin = build_occupancy_grid(
    MAP_SIM['outer'], MAP_SIM['obstacles'],
    X_RANGE, Y_RANGE, cell_size=CELL_SIZE,
    inflate_cells=INFLATE, robot_radius=ROBOT_RADIUS_FT,
)
print(f"Grid built: {grid.shape}, origin (cell-0 center): {origin}")

Grid built: (15, 11), origin (cell-0 center): (-6.0, -5.0)


## 3. BLE connect

In [3]:
ble = get_ble_controller()
ble.connect()
ble.send_command(CMD.PING, "")
print("BLE connected.")

2026-05-13 22:22:54,082 | INFO     |: Looking for Artemis Nano Peripheral Device: c0:42:0c:78:b8:49
2026-05-13 22:22:54,083 | INFO     |: Scanning for device with address: c0:42:0c:78:b8:49, service UUID: d427e7cc-c400-4597-b417-d564e20d6600
2026-05-13 22:23:04,200 | INFO     |: Found 1 device(s) advertising service d427e7cc-c400-4597-b417-d564e20d6600
2026-05-13 22:23:04,201 | INFO     |: Selecting device: 3AE32BFA-4071-EC2D-ED7A-60CCA92DBE18 (name: Artemis BLE)
2026-05-13 22:23:05,208 | INFO     |: Connected to c0:42:0c:78:b8:49
BLE connected.


## 4. Open-loop calibration + control-mode constants
NAV-mode params persist on the Artemis until SET_MODE changes them.

In [4]:
MODE_IDLE    = 3
MODE_MAPPING = 5
MODE_NAV_SEG = 6

TEST_PWM   = 70.0
TEST_SPEED = 1.76    # m/s at TEST_PWM (refine via Phase 2 calibration if needed)

ble.send_command(CMD.SET_MODE, str(MODE_NAV_SEG))
ble.send_command(CMD.UPDATE_ORIENT_PID, "2.5|1.2|0.25")
ble.send_command(CMD.SET_NAV_CALIB,     f"{TEST_PWM}|{TEST_SPEED}")
ble.send_command(CMD.SET_NAV_DIST_MODE, "0")           # time-based
print("NAV_SEG mode armed; PID + calibration + time-mode set.")

NAV_SEG mode armed; PID + calibration + time-mode set.


## 5. Unified log handler
One handler, three message shapes routed by `len(parts)` and the first character:

| Source            | Format                              | Fields |
|---                |---                                  |---|
| NAV log sample    | `T:\|LPWM:\|RPWM:\|AX:\|T2:\|DV:\|YW:` | 7 |
| NAV_DONE ack      | `D:\|S:\|F:\|Y:`                      | 4 (starts with `D`) |
| MAPPING log sample| `T:\|YW:\|T1:\|T2:`                   | 4 (starts with `T`) |

In [5]:
# NAV log buffers — populated by every SEND_LOG response (Arduino sends
# the 7-field T|LPWM|RPWM|AX|T2|DV|YW format in BOTH NAV_SEG and MAPPING).
times             = []
left_pwm_values   = []
right_pwm_values  = []
acc_x_values      = []
tof_2_values      = []
kfpos_values      = []
yaw_values        = []

# NAV_DONE acks (4-field, starts with "D"), keyed by seg_id
last_ack = {}


def reset_nav_buffers():
    times.clear(); left_pwm_values.clear(); right_pwm_values.clear()
    acc_x_values.clear(); tof_2_values.clear()
    kfpos_values.clear(); yaw_values.clear()


def log_handler(sender, data: bytearray):
    """Two message shapes:
      * 7 fields → NAV / MAPPING log sample
      * 4 fields starting with "D" → NAV_DONE ack
    """
    msg = ble.bytearray_to_string(data)
    parts = msg.split("|")

    if len(parts) == 7:
        t    = float(parts[0][2:])
        lpwm = float(parts[1][5:])
        rpwm = float(parts[2][5:])
        ax   = float(parts[3][3:])
        t2   = float(parts[4][3:])
        dv   = float(parts[5][3:])
        yw   = float(parts[6][3:])
        times.append(t)
        left_pwm_values.append(lpwm)
        right_pwm_values.append(rpwm)
        acc_x_values.append(ax)
        tof_2_values.append(t2)
        kfpos_values.append(dv)
        yaw_values.append(yw)

    elif len(parts) == 4 and parts[0][:1] == "D":
        seg    = int(float(parts[0][2:]))
        reason =          parts[1][2:].strip()
        tof_mm = float(parts[2][2:])
        yaw_dg = float(parts[3][2:])
        last_ack[seg] = dict(stop=reason, tof=tof_mm, yaw=yaw_dg)
        print(f"  DONE seg={seg}, stop={reason}, "
              f"tof={tof_mm:.0f} mm, yaw={yaw_dg:+.1f}°")


# Idempotent attach — re-running this cell won\'t raise.
try:
    ble.stop_notify(ble.uuid["RX_STRING"])
except Exception:
    pass
ble.start_notify(ble.uuid["RX_STRING"], log_handler)
print("Unified log handler attached.")


Unified log handler attached.


## 6. `send_segment` with DONE-ack polling
Default timeout bumped to **20 s** — covers worst-case 180° turn + 5 ft
run + tail + BLE latency.

In [6]:
async def send_segment(seg_id, heading_deg, dist_m, timeout_s=20.0, poll_s=0.05):
    """Fire one navigation segment and wait for the NAV_DONE ack.

    MUST be awaited (not called bare) — uses asyncio.sleep so the bleak
    notify callback can fire while we're polling. Using time.sleep here
    blocks the event loop and the ack arrives only after a timeout.

    Returns the ack dict {'stop', 'tof', 'yaw'} on success, or None on timeout.
    """
    last_ack.pop(seg_id, None)
    print(f"-> seg {seg_id}: heading {heading_deg:+.1f}°, dist {dist_m:.3f} m")
    ble.send_command(CMD.SET_NAV_TARGET, f"{heading_deg}|{dist_m}|{seg_id}")
    await asyncio.sleep(0.10)
    ble.send_command(CMD.START_RECORD, "")

    t0 = time.time()
    while seg_id not in last_ack:
        if time.time() - t0 > timeout_s:
            print(f"   !! timeout {timeout_s:.1f}s, no DONE for seg {seg_id}")
            return None
        await asyncio.sleep(poll_s)
    elapsed = time.time() - t0
    ack = last_ack[seg_id]
    print(f"   <- seg {seg_id} done in {elapsed:.2f}s")
    return ack


## 7. Localization framework
- `RealRobot.get_pose()` returns the **commanded target** as both odom and GT,
  so the Lab 10/11 plotter shows the planned waypoint as the green dot.
  The blue dot will then come from the localization belief.
- `perform_observation_loop` is passive: it sends the MAPPING commands and
  reads from the shared `map_*` buffers that the unified handler already fills.
- `Mapper(robot)` ray-traces every cell at construction (12 × 9 × 18 × 18 ≈ 35k rays)
  — expect a 5–10 s pause the first time this cell runs.

In [7]:
WORLD_CFG = os.path.join(str(pathlib.Path(os.getcwd()).parent),
                         "config", "world.yaml")


class RealRobot:
    """Lab 11-compatible robot wrapper for Phase 3.

    We don't use Localization's built-in plot helpers that read get_pose() —
    instead the orchestrator drives `cmdr.plot_gt(...)` for the goal and
    `loc.plot_update_step_data(plot_data=True)` for the belief, matching
    the Lab 10 / inClassDemo style.
    """

    def __init__(self, commander, ble):
        self.world_config  = WORLD_CFG
        self.config_params = load_config_params(self.world_config)
        self.cmdr = commander
        self.ble  = ble

    def get_pose(self):
        # Real robot has no odometry; return zeros (Lab 11 convention).
        return np.array([0.0, 0.0, 0.0]), np.array([0.0, 0.0, 0.0])

    async def perform_observation_loop(self, rot_vel=120):
        """Spin 360° in MAPPING mode, then pull the log via SEND_LOG and
        format 18 ToF readings + bearings for the Bayes update.

        The Arduino SEND_LOG handler emits the 7-field NAV-style log for
        every mode, so we read from the shared `tof_2_values` / `yaw_values`
        buffers — NOT from per-mode buffers.
        """
        _ = rot_vel
        reset_nav_buffers()

        observations_count = self.config_params["mapper"]["observations_count"]

        self.ble.send_command(CMD.SET_MAP_DEGREES, "380")
        self.ble.send_command(CMD.SET_MODE,        str(MODE_MAPPING))
        self.ble.send_command(CMD.UPDATE_ORIENT_PID, "2.5|1.2|0.25")
        self.ble.send_command(CMD.SET_SAMPLE_RATE, "10")
        self.ble.send_command(CMD.SET_DURATION,    "20000")
        self.ble.send_command(CMD.START_RECORD,    "")
        await asyncio.sleep(22)

        self.ble.send_command(CMD.SEND_LOG, "")
        await asyncio.sleep(8)

        # Lab 11 style: in MAPPING mode the Arduino writes -1 to tof_2_buffer
        # between MAP_MEASURE steps and the real ToF only at the stabilized
        # 20° increments. Filtering with (tof > 0) drops every -1 entry and
        # leaves exactly the ~18 stabilized samples — no bearing-binning
        # needed. See ble_arduino.ino:1750–1753.
        tof_2 = np.array(tof_2_values, dtype=float)
        yaws  = np.array(yaw_values,   dtype=float)
        if len(tof_2) == 0 or len(yaws) == 0:
            raise RuntimeError(
                "No mapping samples captured — check Arduino MAPPING FSM, "
                "MODE=5, and that SEND_LOG actually streamed back."
            )

        yaw_unwrapped  = np.degrees(np.unwrap(np.radians(yaws)))
        yaw_continuous = yaw_unwrapped - yaw_unwrapped[0]

        valid_indices    = (tof_2 > 0) & (tof_2 < 6000)
        valid_angles     = np.radians(yaw_continuous[valid_indices])
        valid_distances  = tof_2[valid_indices] + 69.85    # ToF mounting offset (mm)

        if len(valid_distances) < observations_count:
            raise RuntimeError(
                f"Insufficient observation samples: "
                f"got {len(valid_distances)}, expected {observations_count}."
            )

        ranges_in_meters = [d / 1000.0 for d in valid_distances[:observations_count]]
        # ---- Reverse to match the framework's CW scan order. ----
        # The Arduino MAPPING FSM drives IMU yaw POSITIVELY (steps +20° per
        # measurement). With L/R-reversed wiring (INVERT_HEADING), positive
        # IMU yaw corresponds to a CCW physical rotation in world coordinates,
        # so the chronological measurements walk CCW around the robot.
        #
        # The framework's precached views, however, assume the i-th sample
        # is at the cell's heading rotated CW by i*20° (see Mapper.populate_views:
        # bearings = arange(0, 360, 20) + pose[2], with ray dir (cos, -sin)).
        # In addition, the chronological scan starts at +20° (not +0°): the
        # first MAP_MEASURE happens AFTER MAP_TURN reaches map_start_angle +
        # 20°, and the 18th measurement is at +360° (back at start).
        #
        # Reversing turns chrono [m0=+20°, m1=+40°, …, m17=+360°≡+0°] into
        # [new0=+0°, new1=+340°≡−20°≡+20° CW, …, new17=+20° CCW≡+340° CW] —
        # exactly the CW ordering the framework expects.
        ranges_in_meters = ranges_in_meters[::-1]
        sensor_ranges    = np.array(ranges_in_meters)[np.newaxis].T
        sensor_bearings  = np.array(valid_angles[:observations_count][::-1])[np.newaxis].T

        print(f"   scan: {len(valid_distances)} valid ToF samples, "
              f"first range (after CW reorder) = {sensor_ranges[0, 0]:.2f} m")
        return sensor_ranges, sensor_bearings


# Use the Lab 10/11 GUI commander so we can drive plot_gt / plot_bel directly.
gui    = GET_GUI()
cmdr   = gui.launcher.commander
robot  = RealRobot(cmdr, ble)
mapper = Mapper(robot)                # builds the discrete grid + obs_views
loc    = Localization(robot, mapper)  # Localization needs both
loc.init_grid_beliefs()
print("Localization framework ready.")


2026-05-13 22:23:06,642 | INFO     |:  | Number of observations per grid cell: 18
2026-05-13 22:23:06,643 | INFO     |:  | Precaching Views...


/Users/y1hhnn/Desktop/ECE4160/FastRobots-sim-release/localization.py:150: RuntimeWarning: All-NaN slice encountered
  return np.nanmin(distance_intersections_tt), intersections_tt[np.nanargmin(distance_intersections_tt)]


2026-05-13 22:23:07,581 | INFO     |:  | Precaching Time: 0.938 secs
2026-05-13 22:23:07,582 | INFO     |: Initializing beliefs with a Uniform Distribution
2026-05-13 22:23:07,582 | INFO     |: Uniform Belief with each cell value: 0.00051440329218107
2026-05-13 22:23:07,583 | INFO     |: Initializing beliefs with a Uniform Distribution
2026-05-13 22:23:07,584 | INFO     |: Uniform Belief with each cell value: 0.00051440329218107
Localization framework ready.


## 8. Start the plotter
Run this **once** at the start of the session (and again after `STOP_PLOTTER()`
if you ever stop it). Lab 11 uses the same pattern.

In [8]:
START_PLOTTER()

## 9. Helpers: plan + localize_once
- `plan_segments(start_ft, goal_ft)` → list of `Segment` from A*.
- `localize_once()` → single 360° scan + Bayes update. Returns believed
  `(x_ft, y_ft, theta_deg)` for the planner to consume on the next hop.

In [9]:
def plan_segments(start_ft, goal_ft):
    """A* from start to goal (both in ft). Returns list of Segment."""
    sa = world_to_cell(start_ft, origin, CELL_SIZE)
    sb = world_to_cell(goal_ft,  origin, CELL_SIZE)
    cells = astar(grid, sa, sb)
    if not cells:
        raise RuntimeError(f"No path {start_ft} -> {goal_ft}")
    return cells_to_segments(cells, origin, CELL_SIZE)


def plot_planned_path(start_ft, segments):
    """Show the replanned path on the live plotter as a red odom trail."""
    cmdr.plot_odom(start_ft[0] * FT_TO_M, start_ft[1] * FT_TO_M)
    for seg in segments:
        cmdr.plot_odom(seg.p_end[0] * FT_TO_M, seg.p_end[1] * FT_TO_M)


def _wrap_180(deg):
    """Wrap an angle into [-180, 180]."""
    return (deg + 180.0) % 360.0 - 180.0


# ---- Heading convention bridge ------------------------------------------
# Our planner's Segment.heading_deg is "compass": 0° = +y, CW positive.
# The localization framework's cell θ is "math navigation": 0° = +x, CW
# positive — its ray direction is (cos α, -sin α) so α=0° points to +x
# and α=90° points to -y. Mapping between the two is just a 90° offset:
#     framework_angle = user_heading - 90°
# (e.g., user 0° = +y ↔ framework -90°; user 90° = +x ↔ framework 0°).
# Without this offset the framework pins θ to the wrong cell slice and the
# Bayes (x, y) update lands ~90° away from the real wall pattern.
def user_to_framework_angle(user_heading_deg):
    return _wrap_180(user_heading_deg - 90.0)


async def localize_once(current_world_heading_deg):
    """One 360° observation + Bayes update — but only over (x, y).

    We pin the angular dimension to the framework's bin matching
    `current_world_heading_deg` (converted from compass → framework
    convention) by zeroing every other θ slice. The framework's
    update_step then keeps it zero outside that slice, so the argmax
    lands on the correct θ and the believed θ comes back exactly equal
    to the commanded heading.

    Two corrections are essential for the believed (x, y) to match the
    real robot pose:
      1. The user→framework angle conversion above. Pinning to
         `current_world_heading_deg` directly puts the prior on a
         cell slice that is 90° off the robot's true heading.
      2. The chronological CCW-ordered scan is reversed inside
         `perform_observation_loop` so the framework sees a CW-ordered
         scan that matches its precached views.

    Returns (x_ft, y_ft, theta_world_deg). theta_world_deg ≈
    current_world_heading_deg (snapped to the nearest 20° bin).
    """
    # Pin to the framework-angle bin that corresponds to the commanded
    # compass heading. to_map((0, 0, a)) rounds `a` into a valid (ca) bin.
    framework_pin_angle = user_to_framework_angle(current_world_heading_deg)
    _, _, ca_known = loc.mapper.to_map(0.0, 0.0, framework_pin_angle)

    # Build a (x, y, θ)-shaped prior that is uniform over (x, y) on the
    # ca_known slice and exactly zero everywhere else.
    bel_bar = np.zeros_like(loc.bel)
    n_xy = bel_bar.shape[0] * bel_bar.shape[1]
    bel_bar[:, :, ca_known] = 1.0 / n_xy
    loc.bel_bar = bel_bar
    # Mirror into bel so any sanity check sees the right state.
    loc.bel = bel_bar.copy()

    await loc.get_observation_data()
    loc.update_step()
    loc.plot_update_step_data(plot_data=True)   # blue belief dot + framework log

    idx = np.unravel_index(np.argmax(loc.bel), loc.bel.shape)
    x_m, y_m, framework_theta_deg = loc.mapper.from_map(*idx)
    # Convert the framework's θ back to the user's compass convention for
    # the print (caller discards this value and uses commanded heading).
    theta_world_deg = _wrap_180(framework_theta_deg + 90.0)

    print(f"   x,y-only Bayes update — locked to framework θ bin {ca_known} "
          f"(framework {framework_theta_deg:+.1f}° = compass "
          f"{theta_world_deg:+.1f}°, commanded "
          f"{current_world_heading_deg:+.1f}°)")
    return x_m / FT_TO_M, y_m / FT_TO_M, theta_world_deg


## 10. Mission orchestration loop
For each waypoint pair: re-plan from the *believed* pose, execute every
segment, set the robot's target pose (so the plotter draws the green dot
there), then localize. The blue belief dot animates onto the plotter as
each `update_step()` runs.

In [10]:
# ---- L/R-reversed wiring compensation ----------------------------------
INVERT_HEADING = True


# Localize at every N hops. 3 = hops 3, 6, 9 (8 in this map). 1 = every hop.
LOCALIZE_EVERY = 3

# On a front-ToF safety stop, run a rescue localization and re-plan to
# the *same* current target. Capped to avoid infinite loops.
MAX_TOF_RETRIES = 3


def world_to_tx(world_heading_deg, yaw_world_offset_deg):
    """Convert a world-frame heading into the IMU target to send."""
    if INVERT_HEADING:
        return yaw_world_offset_deg - world_heading_deg
    return world_heading_deg - yaw_world_offset_deg


async def run_mission():
    ble.send_command(CMD.RESET_YAW, "")
    await asyncio.sleep(0.3)
    print("Yaw reset; robot is at (-4, -3) facing +y.\n")

    RESET_PLOTTER()
    cmdr.plot_map()

    start_x_m = WAYPOINTS[0][0] * FT_TO_M
    start_y_m = WAYPOINTS[0][1] * FT_TO_M
    cmdr.plot_gt(start_x_m, start_y_m)

    believed_ft           = WAYPOINTS[0]
    yaw_world_offset      = 0.0
    current_world_heading = 0.0     # robot starts physically facing +y (world 0°)
    history               = [(believed_ft, 'start')]
    n_hops                = len(WAYPOINTS) - 1

    for hop in range(1, n_hops + 1):
        target_ft = WAYPOINTS[hop]
        print(f"\n========== Hop {hop}/{n_hops}: "
              f"{believed_ft} → {target_ft} ==========")

        rescued_this_hop = False
        attempts = 0
        hop_succeeded = False
        while not hop_succeeded and attempts <= MAX_TOF_RETRIES:
            if attempts > 0:
                print(f"  -- retry {attempts}/{MAX_TOF_RETRIES} "
                      f"(re-planning to current goal {target_ft}) --")

            segments = plan_segments(believed_ft, target_ft)
            print(f"  Planned {len(segments)} segment(s) "
                  f"(yaw_world_offset = {yaw_world_offset:+.1f}°, "
                  f"current_world_heading = {current_world_heading:+.1f}°):")
            for k, s in enumerate(segments):
                tx = world_to_tx(s.heading_deg, yaw_world_offset)
                print(f"    [{k}] world heading {s.heading_deg:+7.2f}° "
                      f"-> tx {tx:+7.2f}°, "
                      f"dist {s.distance:.2f} ft ({s.distance * FT_TO_M:.3f} m)")

            plot_planned_path(believed_ft, segments)

            ble.send_command(CMD.SET_MODE, str(MODE_NAV_SEG))
            await asyncio.sleep(0.1)

            tof_stopped = False
            for k, seg in enumerate(segments):
                seg_id = hop * 100 + attempts * 10 + k
                tx_heading = world_to_tx(seg.heading_deg, yaw_world_offset)
                ack = await send_segment(seg_id, tx_heading,
                                         seg.distance * FT_TO_M, timeout_s=20.0)
                if ack is None:
                    raise RuntimeError(f"hop {hop} seg {k} timed out")
                if ack['stop'] == 'tof':
                    print(f"  !! hop {hop} seg {k} hit ToF safety stop "
                          f"(tof={ack['tof']:.0f} mm) — rescue-scan + "
                          f"re-plan to SAME goal {target_ft}")
                    tof_stopped = True
                    # Heading update happens BEFORE break — the NAV_TURN
                    # completed, so the robot IS at this segment\'s commanded
                    # heading even though NAV_GO aborted on ToF.
                    current_world_heading = seg.heading_deg
                    break
                # Segment completed normally: orient PID drove the robot
                # to this segment\'s commanded heading. Track it for the
                # next localization\'s yaw_world_offset.
                current_world_heading = seg.heading_deg

            if not tof_stopped:
                hop_succeeded = True
                break

            # Rescue scan — use (x, y) belief, but yaw_world_offset comes
            # from the commanded heading we just set.
            x_ft, y_ft, _ = await localize_once(current_world_heading)
            drift = math.hypot(x_ft - target_ft[0], y_ft - target_ft[1])
            print(f"  rescue belief: ({x_ft:+.2f}, {y_ft:+.2f}) ft "
                  f"(framework θ ignored; using commanded {current_world_heading:+.1f}°) — "
                  f"{drift:.2f} ft from current goal")
            believed_ft      = (x_ft, y_ft)
            yaw_world_offset = current_world_heading   # MAP_START reset IMU=0,
                                                       # world heading there == commanded.
            rescued_this_hop = True
            attempts += 1

        if not hop_succeeded:
            print(f"!! hop {hop} gave up after {MAX_TOF_RETRIES} retries — "
                  f"continuing anyway")

        target_x_m = target_ft[0] * FT_TO_M
        target_y_m = target_ft[1] * FT_TO_M
        cmdr.plot_gt(target_x_m, target_y_m)

        # Scheduled localization
        is_loc_hop = (hop % LOCALIZE_EVERY == 0) or (hop == n_hops)
        if is_loc_hop and not rescued_this_hop:
            print(f"  Localizing at expected pose {target_ft}…")
            x_ft, y_ft, _ = await localize_once(current_world_heading)
            drift = math.hypot(x_ft - target_ft[0], y_ft - target_ft[1])
            print(f"  Belief: ({x_ft:+.2f}, {y_ft:+.2f}) ft "
                  f"(framework θ ignored; using commanded {current_world_heading:+.1f}°) — "
                  f"drift {drift:.2f} ft from expected")
            yaw_world_offset = current_world_heading
            believed_ft      = (x_ft, y_ft)
            history.append((believed_ft, f'hop {hop} (loc)'))
        elif rescued_this_hop:
            history.append((believed_ft, f'hop {hop} (rescued)'))
            print(f"  (skipping scheduled localization — already rescue-scanned)")
        else:
            believed_ft = target_ft
            history.append((believed_ft, f'hop {hop}'))
            print(f"  (skipping localization — every {LOCALIZE_EVERY} hops)")

    return history


## 11. RUN MODE
Flip to `True` and run this cell to start the mission. The live plotter
window will animate green (target) and blue (belief) points at each hop.

In [11]:
cmdr.reset_plotter()
cmdr.plot_map()


In [12]:
RUN = True                  # ← flip to True when ready

if RUN:
    history = asyncio.run(run_mission())
    print("\n=== Mission complete ===")
    for pt, label in history:
        print(f"  {label:>10}: ({pt[0]:+.2f}, {pt[1]:+.2f}) ft")
else:
    print("RUN is False — flip the flag and re-run this cell to start.")

Yaw reset; robot is at (-4, -3) facing +y.


========== Hop 1/8: (-4, -3) → (-2, -1) ==========
  Planned 1 segment(s) (yaw_world_offset = +0.0°, current_world_heading = +0.0°):
    [0] world heading  +45.00° -> tx  -45.00°, dist 2.83 ft (0.862 m)
-> seg 100: heading -45.0°, dist 0.862 m
  DONE seg=100, stop=time, tof=1679 mm, yaw=-52.6°
   <- seg 100 done in 3.14s
  (skipping localization — every 3 hops)

========== Hop 2/8: (-2, -1) → (1, -1) ==========
  Planned 1 segment(s) (yaw_world_offset = +0.0°, current_world_heading = +45.0°):
    [0] world heading  +90.00° -> tx  -90.00°, dist 3.00 ft (0.914 m)
-> seg 200: heading -90.0°, dist 0.914 m
  DONE seg=200, stop=time, tof=2000 mm, yaw=-92.2°
   <- seg 200 done in 2.89s
  (skipping localization — every 3 hops)

========== Hop 3/8: (1, -1) → (2, -3) ==========
  Planned 2 segment(s) (yaw_world_offset = +0.0°, current_world_heading = +90.0°):
    [0] world heading +180.00° -> tx -180.00°, dist 1.00 ft (0.305 m)
    [1] world heading +

Exception: Not connected to a BLE device

## 12. Plot the realized trajectory (after the mission)
Drops the believed positions onto a static matplotlib map for the report.
The live plotter (cell 8) is the in-session view; this is the saved snapshot.

In [ ]:
def plot_history(history):
    from matplotlib.patches import Rectangle
    from astar import polygon_lines

    fig, ax = plt.subplots(figsize=(11, 8))

    occ_x, occ_y = np.where(grid == 1)
    for cx_idx, cy_idx in zip(occ_x, occ_y):
        cx, cy = cell_to_world((cx_idx, cy_idx), origin, CELL_SIZE)
        ax.add_patch(Rectangle(
            (cx - CELL_SIZE / 2, cy - CELL_SIZE / 2),
            CELL_SIZE, CELL_SIZE,
            facecolor='#cccccc', edgecolor='#999999', linewidth=0.4,
        ))

    for (p, q) in polygon_lines(MAP_SIM['outer']):
        ax.plot([p[0], q[0]], [p[1], q[1]], 'k-', linewidth=2)
    for obs in MAP_SIM['obstacles']:
        for (p, q) in polygon_lines(obs):
            ax.plot([p[0], q[0]], [p[1], q[1]], 'k-', linewidth=2)

    wxs, wys = zip(*WAYPOINTS)
    ax.scatter(wxs, wys, c='red', s=140, marker='*', zorder=5, label='planned waypoints')

    if history:
        hxs = [pt[0] for pt, _ in history]
        hys = [pt[1] for pt, _ in history]
        ax.plot(hxs, hys, 'o-', color='tab:blue', linewidth=2, markersize=8,
                label='believed trajectory')
        for i, (pt, _) in enumerate(history):
            ax.annotate(f' {i}', pt, fontsize=10)

    ax.set_aspect('equal')
    ax.set_xlim(X_RANGE[0] - 0.5, X_RANGE[1] + 0.5)
    ax.set_ylim(Y_RANGE[0] - 0.5, Y_RANGE[1] + 0.5)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.set_xlabel('x (ft)')
    ax.set_ylabel('y (ft)')
    ax.legend(loc='upper left')
    ax.set_title('Lab 12 — Phase 3: planned waypoints vs believed trajectory')
    plt.tight_layout()
    out = 'lab12_phase3_run.png'
    plt.savefig(out, dpi=130)
    plt.show()
    print(f"Saved {out}")


if RUN:
    plot_history(history)

## 13. Teardown

In [ ]:
ble.send_command(CMD.STOP_ROBOT, "")
ble.send_command(CMD.SET_MODE, str(MODE_IDLE))
ble.stop_notify(ble.uuid["RX_STRING"])
STOP_PLOTTER()
print("Motors off, notify detached, plotter stopped.")

# ble.disconnect()